# Aşama 17: Gökada düzlemi

2023'te IceCube, **Samanyolu'nun kendisinden** nötrino geldiğini gösterdi: kozmik ışınlar gökada içindeki gazla çarpışıp nötrino üretiyor. Bu, nokta kaynak değil **gökyüzüne yayılmış** bir sinyal: gökada düzlemi boyunca bir bant.

Biz de bunu test ediyoruz. Likelihood aynı; tek fark sinyal yoğunluğu artık bir noktanın etrafında değil, **gökada düzlemi şablonuna** göre dağılıyor. Spektrum γ = 2.7 (gökadadaki kozmik ışınlar için beklenen değer) sabit.

**Önemli bir uyarı:** IceCube'un 2023 sonucu "cascade" denen farklı bir olay türüyle elde edildi. Bizim verimiz "track" (iz) olayları. Gökada merkezi güney gökyüzünde, track olaylarının en zayıf olduğu yerde. Bu yüzden anlamlı bir sinyal çıkmaması **beklenen** sonuç; asıl değer yöntemi uygulamak ve neden zor olduğunu göstermek.

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('15.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')
import os, time
from scipy import stats
N_CEKIRDEK = os.cpu_count()
sig0 = lambda p: max(0.0, ps.sigma(p))
alanlar, sureler, smearingler = ps.etkin_alanlari_yukle(DATA), ps.canli_sureler_yukle(DATA), ps.smearingleri_yukle(DATA)

In [ ]:
M = ps.SezonluModel(ev, alanlar, sureler, smearingler)
sablonlar = {
    'Bant (σ_b = 5°)': dict(sigma_b=5.0, sigma_l=None),
    'Merkeze doğru parlayan (σ_b = 5°, σ_l = 30°)': dict(sigma_b=5.0, sigma_l=30.0),
}
modeller = {ad: ps.GalaksiDuzlemiLikelihood(M, gamma=2.7, **p) for ad, p in sablonlar.items()}

In [ ]:
# Sinyalin IceCube'dan nasıl göründüğü: şablon × dedektör kabulü
G = modeller['Merkeze doğru parlayan (σ_b = 5°, σ_l = 30°)']
ra_h, dec_h = ps.gok_izgarasi(1.0, -89.5, 89.5)
s0 = M.sezonlar[-1]
k = np.clip(np.digitize(dec_h, G.dec_kutu) - 1, 0, 179)
gorunen = G._sablon(ra_h, dec_h) * G.S_norm[s0][1][k]
gx = lambda r: -(np.radians(r) - np.pi)
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5), subplot_kw={'projection': 'mollweide'})
for ax, deger, baslik in [(axs[0], G._sablon(ra_h, dec_h), 'Gökada şablonu (ne kadar nötrino üretiliyor)'),
                          (axs[1], gorunen / gorunen.max(), 'Şablon × IceCube track kabulü (ne kadarını görüyoruz)')]:
    o = np.argsort(deger)
    ax.scatter(gx(ra_h[o]), np.radians(dec_h[o]), c=deger[o], s=3, cmap='viridis', lw=0)
    ax.set_title(baslik, fontsize=10); ax.grid(alpha=0.3); ax.set_xticklabels([])
plt.savefig(RESULTS / 'asama17_sablon.png', dpi=150, bbox_inches='tight')
plt.show()

**Grafiği okurken:** Soldaki şablonda gökada merkezi (güneyde) en parlak. Sağda ise neredeyse hiçbir şey kalmıyor: track olaylarında güney gökyüzündeki hassasiyet çok düşük, görebildiğimiz kısım gökada düzleminin kuzeydeki sönük parçası. Testin zorluğunu tek bakışta gösteren bir grafik.

## Fit ve anlamlılık

⏱️ Şablon başına 500 karıştırma, birkaç dakika.

In [ ]:
fitler = []
for ad, G in modeller.items():
    ns, ts = G.fit()
    dosya = RESULTS / f"asama17_arka_{'bant' if 'Bant' in ad else 'merkez'}.npy"
    if dosya.exists():
        ts_a = np.load(dosya)
    else:
        rng = np.random.default_rng(1700)
        ts_a = np.array([G.fit(G.karistir(rng))[1] for _ in (tqdm(range(500)) if tqdm else range(500))])
        np.save(dosya, ts_a)
    p_mc = (np.sum(ts_a >= ts) + 1) / (len(ts_a) + 1)
    fitler.append(dict(sablon=ad, ns=ns, TS=ts, p_MC=p_mc, sigma=sig0(p_mc),
                       TS_arka_medyan=np.median(ts_a), TS_arka_90=np.percentile(ts_a, 90)))
tablo17 = pd.DataFrame(fitler).set_index('sablon')
print(f"İki şablon düzeltmesiyle en iyi: {sig0(2 * tablo17['p_MC'].min()):.2f}σ")
tablo17.to_csv(RESULTS / 'asama17_galaksi.csv')
tablo17

## Yorumlarken

- **Anlamlı bir sinyal yoksa** bu, track olaylarının bu ölçüm için doğru araç olmadığını gösterir; IceCube'un 2023 keşfinin neden cascade olaylarıyla yapıldığını açıklar. Cascade'lerin yön belirsizliği büyük (~10°) ama güney gökyüzünü ve düşük enerjileri çok daha iyi görüyorlar.
- **Zayıf bir fazlalık varsa** dikkatli olmak gerek: şablonun kendisi (σ_b, σ_l seçimleri) basit; IceCube fiziksel modellerden türetilmiş şablonlar (π⁰ ve KRA modelleri) kullanıyor.
- Bu aşama, nokta kaynak analizinin **yayılmış kaynaklara** da genelleştirilebildiğini gösteriyor: aynı likelihood, sadece farklı bir sinyal şekli.